<center><h1>Document Simplifier (fine-tuned)</h1></center>

**IAT 360 Group 17:** Manroop Sandhu, Sean Penner, Kent Nguyen, Marielle Reyes Lao

So after the last round of feedback we talked to Yalcin and Minoo, and it turns out this is actually an option 1 project now (we're training a model), and RAG just isn't really the right tool for this kind of task, so that whole retrieval part is gone now. What we're actually doing: **sentence-level text simplification** (give it one complicated sentence, it gives back a simpler one that still means the same thing), fine-tuning Qwen2.5-7B-Instruct with LoRA so it still fits on the free Colab GPU.

Still a bunch of `# TODO`s in here, mostly around how much training data/how many epochs we can actually get away with before Colab kicks us off, but the whole pipeline runs start to finish.

**Before running:** Runtime > Change runtime type > T4 GPU


## Plan
1. **Train** Qwen2.5-7B-Instruct with LoRA on WikiAuto (complex/simple Wikipedia sentence pairs), so it learns to simplify sentences.
2. **Evaluate** with SARI (checks our output against ~10 human reference simplifications per sentence) and Flesch-Kincaid grade level (how easy it is to read), comparing the fine-tuned model to the same model before fine-tuning so we actually know if training helped.
3. **Apply it** to a real document: pull out the text, split it into sentences, simplify each one, put it back together next to the original with the page number.

## Data
- **Training:** WikiAuto, through the `GEM/wiki_auto_asset_turk` dataset on Hugging Face, roughly 480k complex/simple sentence pairs from Wikipedia and Simple Wikipedia.
- **Evaluation:** ASSET, which turns out to just be the `test_asset` split of that same dataset (359 sentences, about 10 human reference simplifications each). Not training on this part, only testing on it.
- **Document testing:** same 4 public documents as before (ICBC guide, SFU policy, passport form, property tax info). Not uploading these here, keeping PDFs in `data/` (gitignored).


## Splitting up the work (again)

Since we're not building the chatbot/RAG thing anymore this probably needs to get split up differently than we originally planned. Haven't actually figured this out as a group yet, just listing out what pieces exist so we can divide it up:

- loading the WikiAuto/ASSET data and checking it looks right, that part's basically done below already
- actually running the fine-tune, which is the slow part, could be a couple hours even with LoRA. sean maybe runs this one since he's got his own GPU and we remember free Colab timing out on us during the CV project?
- the eval part (base model vs fine-tuned, SARI + grade level) needs someone to run it and actually write up what the numbers mean
- hooking it up to a real PDF + the Gradio demo is kind of its own chunk
- and the 40 examples per document thing is probably still one person per document like before, just more of them now

___ / ___ / ___ / ___ can fill in who's actually doing what here, not deciding that part ourselves


In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets evaluate sacrebleu sacremoses textstat pypdf gradio


In [ ]:
import os
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from pypdf import PdfReader
import evaluate
import textstat

print('GPU available:', torch.cuda.is_available())


## Settings

TODO: `TRAIN_SAMPLES` and `NUM_EPOCHS` are just starting guesses, raise them if training finishes with time to spare.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

TRAIN_SAMPLES = 5000   # TODO: WikiAuto has ~480k pairs, we're only using a slice of it to start
NUM_EPOCHS = 1         # TODO: bump this up if training is faster than expected
TOP_K_TEST = 50        # how many ASSET test sentences we evaluate on (the full test set is 359)

SIMPLIFY_SYSTEM_PROMPT = (
    'Rewrite the sentence in simpler, plainer English. '
    'Use everyday words and shorter sentences where you can, but keep the meaning exactly the same.'
)


## 1. Load the training data (WikiAuto)

In [ ]:
wiki_auto = load_dataset('GEM/wiki_auto_asset_turk', split='train')
wiki_auto = wiki_auto.shuffle(seed=42).select(range(TRAIN_SAMPLES))

print(f'{len(wiki_auto)} training pairs')
print('source:', wiki_auto[0]['source'])
print('target:', wiki_auto[0]['target'])


## 2. Load the evaluation data (ASSET)

This is the `test_asset` split of the same dataset. Each sentence comes with a list of human reference simplifications, which is what SARI compares our output against.

In [ ]:
asset_test = load_dataset('GEM/wiki_auto_asset_turk', split='test_asset')
asset_test = asset_test.select(range(TOP_K_TEST))

print(f'{len(asset_test)} test sentences')
print('source:', asset_test[0]['source'])
print(f"{len(asset_test[0]['references'])} reference simplifications, for example:")
print(asset_test[0]['references'][0])


## 3. Turn each training pair into a chat-formatted example

We use Qwen's own chat template so training examples look like the prompts we'll actually use later.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def format_example(example):
    messages = [
        {'role': 'system', 'content': SIMPLIFY_SYSTEM_PROMPT},
        {'role': 'user', 'content': example['source']},
        {'role': 'assistant', 'content': example['target']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

wiki_auto = wiki_auto.map(format_example)
print(wiki_auto[0]['text'])


## 4. Load Qwen2.5-7B-Instruct in 4-bit

7B is too big to fully fine-tune on a free T4, so we load it in 4-bit (QLoRA style) and only train small LoRA adapter layers on top. TODO: if this still runs out of memory, try a smaller Qwen model (like the 1.5B or 3B version) instead.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)


## 5. Add LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Fine-tune

Using `trl`'s `SFTTrainer` since it handles a lot of the LoRA + tokenizing boilerplate for us.

In [ ]:
training_args = SFTConfig(
    output_dir='qwen-simplifier-lora',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    dataset_text_field='text',
    max_seq_length=512,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=wiki_auto,
)

trainer.train()


In [ ]:
model.save_pretrained('qwen-simplifier-lora-final')


## 7. Compare base model vs fine-tuned model

Since the LoRA adapter can be turned on and off on the same model, we can generate with it disabled (base Qwen) and enabled (our fine-tuned version) and compare them fairly.

In [ ]:
def simplify(sentence, use_adapter=True):
    messages = [
        {'role': 'system', 'content': SIMPLIFY_SYSTEM_PROMPT},
        {'role': 'user', 'content': sentence},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    if use_adapter:
        output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    else:
        with model.disable_adapter():
            output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):].strip()

# quick sanity check on one sentence
example_sentence = asset_test[0]['source']
print('Original: ', example_sentence)
print('Base:     ', simplify(example_sentence, use_adapter=False))
print('Fine-tuned:', simplify(example_sentence, use_adapter=True))


In [ ]:
sari = evaluate.load('sari')

def evaluate_model(use_adapter, dataset=asset_test):
    sources = [ex['source'] for ex in dataset]
    references = [ex['references'] for ex in dataset]
    predictions = [simplify(s, use_adapter=use_adapter) for s in sources]

    sari_score = sari.compute(sources=sources, predictions=predictions, references=references)['sari']
    grade_levels = [textstat.flesch_kincaid_grade(p) for p in predictions if p.strip() != '']
    avg_grade = sum(grade_levels) / len(grade_levels)
    return {'sari': sari_score, 'avg_grade_level': avg_grade, 'predictions': predictions}

base_results = evaluate_model(use_adapter=False)
tuned_results = evaluate_model(use_adapter=True)

print(f"Base model:       SARI {base_results['sari']:.2f}, avg grade level {base_results['avg_grade_level']:.1f}")
print(f"Fine-tuned model: SARI {tuned_results['sari']:.2f}, avg grade level {tuned_results['avg_grade_level']:.1f}")


TODO: once this runs, write a sentence or two here about whether fine-tuning actually helped (higher SARI = better simplification, lower grade level = easier to read).

## 8. Apply it to a real document

Splitting into sentences the simple way, on punctuation, same as the segmentation step from lecture.

In [ ]:
def get_pages(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ''
        pages.append({'page': i + 1, 'text': text})
    return pages

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s for s in sentences if s.strip() != '']

def simplify_page(page_text):
    sentences = split_sentences(page_text)
    simplified = [simplify(s, use_adapter=True) for s in sentences]
    return ' '.join(simplified)

# TODO: upload a test PDF and try this
# from google.colab import files
# uploaded = files.upload()
# pdf_path = list(uploaded.keys())[0]
# pages = get_pages(pdf_path)
# print('Original page 1:', pages[0]['text'][:500])
# print('Simplified page 1:', simplify_page(pages[0]['text'])[:500])


## 9. Gradio demo

In [ ]:
import gradio as gr

def run_demo(pdf_file):
    if pdf_file is None:
        return 'Upload a PDF to get started.'
    pages = get_pages(pdf_file.name)
    output = ''
    for page in pages:
        output += f"--- page {page['page']} ---\n{simplify_page(page['text'])}\n\n"
    return output

demo = gr.Interface(
    fn=run_demo,
    inputs=gr.File(label='Upload a PDF', file_types=['.pdf']),
    outputs=gr.Textbox(label='Simplified document', lines=20),
    title='Document Simplifier (Group 17)',
)

# TODO: once the pipeline above works end to end:
# demo.launch(share=True)


## Test examples (one document per person, again)

Probably still makes sense to keep the one-document-per-person split from before, just 40 examples each now instead of 20 (160 total). Pull sentences straight from the actual document and see how the model does, this also doubles as a check on whether training on Wikipedia sentences actually generalizes to government/legal writing, which reads pretty differently.

| Document | Who's doing it | 40 examples pulled? |
|---|---|---|
| ICBC Learn to Drive Smart guide | ___ | not yet |
| SFU Policy S10.01 | ___ | not yet |
| Canadian adult passport application | ___ | not yet |
| BC annual property tax info | ___ | not yet |


## TODO list
- [ ] figure out `TRAIN_SAMPLES` and `NUM_EPOCHS` once we see how long one run actually takes (email Minoo about a remote GPU if Colab keeps cutting us off)
- [ ] actually run the base vs fine-tuned comparison and write a couple sentences on whether it helped
- [ ] everyone pulls their 40 example sentences
- [ ] try it on a real uploaded PDF, not just the ASSET sentences
- [ ] check if any of our 4 documents need OCR (scanned forms)
- [ ] keep `data/` (PDFs) out of git
- [ ] mention the dataset licenses somewhere in the final report, WikiAuto is CC BY-NC 3.0 and ASSET is CC BY-NC 4.0, both non-commercial which is fine for a class project
- [ ] decide who's doing which part from the list above
